In [40]:
print("hello world")

hello world


## Project 4: Multi-Step Workflow with LangGraph.

#### The goal is a travel planner where a graph coordinates research (finding attractions), budget (calculating costs), and itinerary (producing a plan). It can loop back if the budget is insufficient and includes a human‑in‑the‑loop step for final approval.

In [41]:
# !pip install langgraph

1. import required libraries

In [42]:
from typing import List, Annotated, TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import AzureChatOpenAI
import operator



2. define the state

In [43]:
class TravelState(TypedDict):
    destination: str
    interests: List[str]
    budget: float
    itinerary: str
    research_notes: Annotated[List[str], operator.add]
    budget_approved: bool
    messages: List[dict]
    retry_count: int



3. build graph

In [44]:
workflow = StateGraph(TravelState)

4. define the model

In [45]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

az_aoai_key = os.getenv("az_aoai_key")
az_model = os.getenv("az_model")
az_endpoint = os.getenv("az_endpoint")
api_ver = os.getenv("api_ver")



chat_model = AzureChatOpenAI(
    api_key=az_aoai_key,  # type: ignore
    api_version=api_ver,
    model=az_model,
    azure_endpoint=az_endpoint,
    temperature=0.01
)


5-1. node_1: research attraction

In [46]:
def research_node(state: TravelState) -> dict:  # type: ignore
    print(">>> running research node")
    destination = state["destination"]
    # interest = state["interests"]
    interests = ", ".join(state.get("interests", []))   # type:ignore
    research_prompt = f"List 3 attractions in {destination} related to {interests}; keep in short and concise."
    response = chat_model.invoke(research_prompt)
    notes = [line.strip() for line in response.content.split("\n") if line.strip()]  # type: ignore
    if not notes:
        notes = [f"Explore popular attractions in {destination}"]    # type: ignore
    print(f">>> Research Notes: {notes}")
    return {f"\n\tBot Research Notes:": notes}  # type: ignore



5-2. node_2: budget calculation

In [47]:
def budget_node(state: TravelState) -> dict:
    print(">>> running budget node")
    estimated_cost = len(state["research_notes"]) * 750  # Example cost calculation
    if estimated_cost <= state["budget"]:
        print(f">>> Estimated cost: {estimated_cost} is within the Budget: {state['budget']}.")
        return {"budget_approved": True}    # type: ignore
    else:
        print(f">>> Estimated cost: {estimated_cost} exceeds the Budget: {state['budget']}.")
        return {"budget_approved": False}   # type: ignore
    


5-3. node_3: human approval

In [48]:
def human_approval_node(state: TravelState) -> dict:    # type: ignore
    print(">>> running human approval node")
    retry_count = state.get("retry_count", 0) + 1   # type: ignore
    print("\n" + "="*80)
    print(f"Proposed Destination {state['destination']} based on research notes: {state['research_notes']}.")
    print("\nProposed Itinerary:")
    print(state.get("itinerary", "No itinerary generated yet."))   # type: ignore
    print("="*80)
    # ask for human approval
    user_input = input("Budget exceeds the limit. Do you want to proceed with the current plan? (yes/no): ").strip().lower()
    approved = user_input == "yes"
    # if user_input.lower() == "yes":
    #     return {"budget_approved": True}    # type: ignore
    # else:
    #     return {"budget_approved": False}   # type: ignore
    if retry_count >= 3:
        print("Maximum retry attempts reached. Ending workflow.")
        approved = True
    print(f">>> Retry count: {retry_count}, Approved: {approved}")
    return {"budget_approved": approved, "retry_count": retry_count}    # type: ignore
        # return {"budget_approved": False}   # type: ignore

5-4. node_4: Itineray node to generate actual plan

In [49]:
def itinerary_node(state: TravelState) -> dict: # type: ignore
    print(">>> running itinerary node")
    itinerary_prompt = f"Create a 1-day itinerary for {state['destination']} based on these attraction: {state['research_notes']} under Budget: {state['budget']}."
    response = chat_model.invoke(itinerary_prompt)
    itinerary_text = response.content.strip()  # type: ignore
    print(">>> Itinerary Generated")
    return {"itinerary": itinerary_text}   # type: ignore


5-5. condition after adding human approval

In [50]:
def after_approval(state: TravelState) -> str:    # type: ignore
    """Conditional edge after human approval."""
    print(">>> running after approval node")
    if state.get("budget_approved", False):   # type: ignore
        return "end"
    else:
        return "research"

6. add theses notes to workflow

In [51]:
workflow.add_node("research", research_node)    # type: ignore
workflow.add_node("budget", budget_node)    # type: ignore
workflow.add_node("human_approval", human_approval_node)    # type: ignore
workflow.add_node("itinerary", itinerary_node)    # type: ignore


6_2. adding conditional edge i.e. conditional function to decide next step after budget

In [52]:
def should_continue(state: TravelState) -> str:
    """After budget node: go to research if not approved, else itinerary."""
    if not state["budget_approved"] == True:   # type: ignore
        return "research"
    else:
        return "itinerary"  # type: ignore


In [53]:
def budget_router(state: TravelState) -> str:
    if state["budget_approved"]:
        return "itinerary"
    else:
        return "research"

6_3. adding edges to workflow

In [54]:
workflow.set_entry_point("research")    # type: ignore
workflow.add_edge("research", "budget")
workflow.add_conditional_edges("budget", budget_router, {"research": "research", "itinerary": "itinerary"})

# workflow.add_conditional_edges("budget", should_continue, {"research": "research", "itinerary": "itinerary"})
workflow.add_edge("itinerary", "human_approval")
workflow.add_conditional_edges("human_approval", after_approval, {"end": END, "research": "research"})    # type: ignore
workflow.add_edge("budget", "itinerary")
# workflow.add_node("itinerary", lambda state: {"itinerary": f"Final plan for {state['destination']}!"})    # type: ignore
workflow.add_edge("itinerary", END)



7. compile and run the workflow/graph

In [55]:
app = workflow.compile()    # type: ignore

In [56]:
if __name__ == "__main__":
    init_state = {      # type: ignore
        "destination": "Digha",
        "interests": ["beaches", "seafood", "sunsets", "boat ride"],
        "budget": 500,
        "research_notes": [],
        "budget_approved": False,
        "messages": []
        }


In [57]:
result = app.invoke(init_state) # type: ignore
print("\n" + "="*80)
print("FINAL ITINERARY:")
print(f"Itinerary: {result['itinerary']}")
print("="*80)


>>> running research node
>>> Research Notes: ['1. **New Digha Beach** - Perfect for serene sunsets and fresh seafood stalls.', '2. **Udaipur Beach** - Offers tranquil boat rides and picturesque views.', '3. **Shankarpur Beach** - Known for its fishing harbor and sunrise seafood markets.']
>>> running budget node
>>> Estimated cost: 0 is within the Budget: 500.
>>> running itinerary node
>>> Itinerary Generated
>>> running human approval node

Proposed Destination Digha based on research notes: [].

Proposed Itinerary:
Creating a 1-day budget-friendly itinerary for Digha under ₹500 is absolutely possible! Since you haven't listed specific attractions, I'll include some of the most popular and must-visit spots in Digha. Here's a well-planned itinerary:

---

### **1-Day Digha Itinerary (Budget: ₹500)**

#### **Morning**
1. **Sunrise at New Digha Beach**  
   - Start your day early by enjoying the serene sunrise at New Digha Beach. It's a peaceful and refreshing experience.  
   - **Cost

## 🎯 Key Takeaways from Project 4 (LangGraph Workflow)

You've just built a **stateful, conditional, human-in-the-loop graph** – the foundation of production‑grade agents. Here's what you mastered:

---

### 1. **Graphs are the orchestrator for complex logic**
- Unlike linear chains (LCEL), graphs allow **loops, branches, and pauses**.  
- Essential for real‑world tasks like iterative planning, approval flows, and error recovery.

### 2. **State is a typed shared dictionary**
- Defined with `TypedDict`, it holds everything the workflow knows.  
- Use `Annotated[list, operator.add]` to **auto‑append** to lists (instead of overwriting).  
- All nodes read and update the same state object.

### 3. **Nodes are plain Python functions**
- Each node takes the current state and returns a **partial update** (only the keys it changes).  
- LangGraph merges the update back into the state.

### 4. **Conditional edges enable dynamic routing**
- A router function reads the state and returns the **name of the next node**.  
- This allows loops (`research → budget → research`) and branching (`budget → itinerary` if approved).

### 5. **Human‑in‑the‑loop is built with a node that pauses**
- The `human_approval` node waits for user input.  
- After approval, the graph either proceeds or loops back (e.g., re‑plan).  
- **Retry counters** prevent infinite loops – a production necessity.

### 6. **No direct edges when using conditionals**
- Adding both `add_edge` and `add_conditional_edges` from the same node **breaks state propagation**.  
- Choose one routing method per node.

### 7. **State persistence (checkpointing) is separate**
- Though we didn't add it in Project 4, LangGraph supports `MemorySaver` or `SqliteSaver` to pause/resume workflows.  
- Human‑in‑the‑loop becomes truly powerful when you can stop, wait for days, and resume exactly where you left off.

### 8. **LangGraph is not just for agents – it's for any process**
- You can model refund approvals, multi‑step research, data validation pipelines, or even CI/CD workflows.  
- The same pattern works for any multi‑step process with decisions.

---

## 🧠 Mental model for LangGraph

> **State** (memory) → **Nodes** (actions) → **Edges** (routing) → **Conditional edges** (intelligence) → **Human nodes** (supervision)

---

## ✅ What you can now build

- **Approval workflows** (expense requests, content moderation)  
- **Iterative research agents** that refine their search based on budget or time  
- **Customer support triage** (branch to different departments, escalate to human)  
- **Multi‑turn planning systems** that loop until a constraint is met

---
